In [11]:
import nltk
import re
import math
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer

# --- SETUP ---
def initialize_nltk():
    resources = ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'averaged_perceptron_tagger_eng']
    for res in resources:
        nltk.download(res, quiet=True)
    return set(stopwords.words('english')), WordNetLemmatizer()

def get_wordnet_pos(word):
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ, "N": wordnet.NOUN, "V": wordnet.VERB, "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)

def clean_text(text, stop_words, lemmatizer):
    tokens = word_tokenize(text)
    cleaned = []
    for t in tokens:
        t_clean = re.sub(r"[^\w\s]", "", t).lower().strip()
        if not t_clean or re.search(r'\d', t_clean) or t_clean in stop_words:
            continue
        lemma = lemmatizer.lemmatize(t_clean, get_wordnet_pos(t_clean))
        cleaned.append(lemma)
    return cleaned

# --- MATH ---
def calculate_tfidf(all_docs):
    vocab = sorted(set([t for doc in all_docs for t in doc]))
    N = len(all_docs)
    tfs = []
    for doc in all_docs:
        tfs.append({w: doc.count(w) for w in vocab if w in doc})

    idf = {w: math.log((N + 1) / (sum(1 for doc in all_docs if w in doc) + 1)) + 1 for w in vocab}

    vectors = []
    for counts in tfs:
        vectors.append({w: counts.get(w, 0) * idf[w] for w in vocab})
    return vectors

def get_similarity(vec_a, vec_b):
    dot = sum(vec_a.get(k, 0) * vec_b.get(k, 0) for k in vec_a if k in vec_b)
    norm_a = math.sqrt(sum(v**2 for v in vec_a.values()))
    norm_b = math.sqrt(sum(v**2 for v in vec_b.values()))
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0

# --- MAIN ---
if __name__ == "__main__":
    stop_words, lemmatizer = initialize_nltk()

    # Load requirements
    with open('requirements-3nfr-60fr.txt', 'r', encoding='utf-8') as file:
        lines = [line.strip() for line in file if line.strip()]

    nfr_raw = lines[:3]
    # Use simple list for FRs to maintain FR1, FR2 numbering
    fr_raw = lines[3:]

    # Clean
    nfr_docs = [clean_text(t, stop_words, lemmatizer) for t in nfr_raw]
    fr_docs = [clean_text(t, stop_words, lemmatizer) for t in fr_raw]

    # Vectorize
    all_vecs = calculate_tfidf(nfr_docs + fr_docs)
    nfr_vecs = all_vecs[:3]
    fr_vecs = all_vecs[3:]

    # THRESHOLDING LOGIC
    THRESHOLD = 0.12

    # print("Requirement,NFR1,NFR2,NFR3")
    with open("test_output.txt", "w") as f:
        for i, fr_v in enumerate(fr_vecs):
            # Calculate raw scores
            sims = [get_similarity(fr_v, n_v) for n_v in nfr_vecs]

            # Convert to Binary (1 if > threshold, else 0)
            binary_row = [1 if s >= THRESHOLD else 0 for s in sims]

            # Formatting output exactly like your target
            line = f"FR{i+1},{binary_row[0]},{binary_row[1]},{binary_row[2]}"
            # print(line)
            f.write(line + "\n")
